# 2. Analyse des éléments de symétrie

Dans cette section, nous analysons l'effet de trois opérations de symétrie ponctuelle distinctes (en excluant l'identité) appartenant au groupe d'espace $P\bar{4}m2$. Conformément aux consignes du projet, nous illustrons chacune de ces opérations sur un atome différent de la maille de notre matériau : l'Aluminium (Al), le Gallium (Ga) et l'Azote (N). 

Afin de mieux visualiser les opérations de symétrie, nous avons représenté ces dernières en 3D grâce à JSmol. Cet outil de dessin ne permettant pas de visualiser les images directement sur VScode ou Github, nous avons également inséré des captures d'écran des résultats, pour qu'ils soient visibles plus facilement. 
Ces visualisations utilisent un code couleur pour identifier les atomes : gris clair pour l'Aluminium, vert clair pour le Gallium et bleu clair pour l'Azote. De plus, l'atome sur lequel l'opération de symétrie est appliquée est mis en évidence en rouge, et nous avons également illustré le plan de symétrie pour la rotation sur l'Aluminium. 

De plus, puisque nous étudions uniquement des symétries ponctuelles, le vecteur de translation $\tau$ est nul pour les 3 symétries.

#### Imports et récupération de la structure, des coordonnées et des opérations de symmétrie

In [1]:
import numpy as np
from pymatgen.ext.matproj import MPRester
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from IPython.display import display
from ipywidgets import Layout
from jupyter_jsmol.pymatgen import quick_view

API_KEY = "vKJsFu0jdhLy7CJj5Mwar6S68kxgMc3n"
MP_ID = "mp-1008556"

with MPRester(API_KEY) as mpr:
    structure = mpr.get_structure_by_material_id(MP_ID)

# Passage en maille conventionnelle pour l'analyse des symétries
analyzer = SpacegroupAnalyzer(structure)
conv_struc = analyzer.get_conventional_standard_structure()
analyze_struc = SpacegroupAnalyzer(conv_struc)

sym = analyze_struc.get_symmetry_operations()

# Identification des coordonnées d'un atome de chaque espèce
sites = conv_struc.sites
Al_coord = [site.frac_coords for site in sites if site.specie.symbol == 'Al'][0]
Ga_coord = [site.frac_coords for site in sites if site.specie.symbol == 'Ga'][0]
N_coord =  [site.frac_coords for site in sites if site.specie.symbol == 'N'][0]

# Sélection des 3 opérations
S1 = None; S2 = None; S3 = None
for op in sym:
    xyz = op.as_xyz_str().replace(" ", "")
    if xyz == "-x,-y,z":
        S1 = op  # Rotation d'ordre 2
    elif xyz == "x,-y,z" or xyz == "-x,y,z":
        S2 = op  # Plan Miroir
    elif xyz == "-y,x,-z" or xyz == "y,-x,-z":
        S3 = op  # Roto-inversion

Al_sym = S1.operate(Al_coord) # Rotation d'ordre 2 sur l'atome Al
Ga_sym = S2.operate(Ga_coord) # Plan Miroir sur sur l'atome Ga
N_sym = S3.operate(N_coord) # Roto-inversion sur sur l'atome N 


### 2.1 Rotation d'ordre 2 sur l'Aluminium (Al)

La première opération étudiée est une rotation d'ordre 2 ($C_2$) autour de l'axe $z$, appliquée à l'atome d'Aluminium. Mathématiquement, cette opération transforme les coordonnées d'un point $(x,y,z)$ en $(-x,-y,z)$. Elle est décrite par l'équation matricielle suivante : 

$$\begin{pmatrix} x' \\ y' \\ z' \end{pmatrix} = \begin{pmatrix} -1 & 0 & 0 \\ 0 & -1 & 0 \\ 0 & 0 & 1 \end{pmatrix} \begin{pmatrix} x \\ y \\ z \end{pmatrix} + \begin{pmatrix} 0 \\ 0 \\ 0 \end{pmatrix}$$

Le déterminant de la matrice de rotation vaut $1$, il s'agit donc d'une opération de première espèce (rotation propre) qui conserve la chiralité.

Lorsqu'on applique cette rotation à la position de l'atome d'Aluminium dans la maille, celui-ci est envoyé sur une position cristallographiquement équivalente du réseau. En effet, les coordonnées obtenues $(-0.5, -0.5, 0.5)$ sont équivalentes à $(0.5, 0.5, 0.5)$ modulo une translation de vecteur de réseau. Cela confirme que cette rotation appartient bien au groupe de symétrie du cristal. En effet, cette transformation géométrique préserve parfaitement la structure de la maille.

In [2]:
print("--- OPÉRATION DE SYMÉTRIE 1 (Rotation) --- \n")
print(f"L'atome de départ est l'Aluminium (Al) aux coordonnées : {Al_coord}\n")
print(f"Application de l'opération {S1.as_xyz_str()} :\n")
print(S1.rotation_matrix, " * ", Al_coord, " + ", S1.translation_vector)
print(f"\nL'atome d'arrivée a pour coordonnées : {Al_sym}\n")

# Visualisation avec JSmol
view1 = quick_view(conv_struc)
display(view1)

view1.script("reset")
view1.script("background white")
view1.script("unitcell on")
view1.script("axes on")
view1.script("spacefill 23%")
view1.script("wireframe 0.08")
view1.script("zoom 140")

view1.script("select aluminum; color lightgray")
view1.script("select gallium; color lightgreen")
view1.script("select nitrogen; color lightblue")

view1.script("draw delete")
view1.script("draw symop 5 {atomno=1}")
view1.script("select atomno=1; color red; spacefill 0.5")

--- OPÉRATION DE SYMÉTRIE 1 (Rotation) --- 

L'atome de départ est l'Aluminium (Al) aux coordonnées : [0.5 0.5 0.5]

Application de l'opération -x, -y, z :

[[-1.  0.  0.]
 [ 0. -1.  0.]
 [ 0.  0.  1.]]  *  [0.5 0.5 0.5]  +  [0. 0. 0.]

L'atome d'arrivée a pour coordonnées : [-0.5 -0.5  0.5]



JsmolView(layout=Layout(align_self='stretch', height='400px'))

#### Illustration de la rotation d'ordre 2

![Rotation](Images_symmetry/rotation.png)


### 2.2 Réflexion (Plan Miroir) sur le Gallium (Ga)

La deuxième opération étudiée est une réflexion (plan miroir $m$) par rapport au plan $xz$, appliquée à l'atome de Gallium. Mathématiquement, cette opération transforme les coordonnées d'un point $(x,y,z)$ en $(x,-y,z)$. Elle est décrite par l'équation matricielle suivante :  

$$\begin{pmatrix} x' \\ y' \\ z' \end{pmatrix} = \begin{pmatrix} 1 & 0 & 0 \\ 0 & -1 & 0 \\ 0 & 0 & 1 \end{pmatrix} \begin{pmatrix} x \\ y \\ z \end{pmatrix} + \begin{pmatrix} 0 \\ 0 \\ 0 \end{pmatrix}$$

Dans ce cas-ci, le déterminant de cette matrice vaut $-1$, il s'agit donc d'une opération de deuxième espèce, qui inverse la chiralité.

Lorsque cette réflexion est appliquée à la position de l'atome de Gallium dans la maille, celui-ci reste invariant. En effet, l'atome étudié est situé sur le plan miroir (coordonnées $(0,0,0)$), ce qui explique qu'il ne soit pas déplacé par cette transformation. Cela confirme que ce plan miroir appartient aux éléments de symétrie du cristal.

In [3]:
print("--- OPÉRATION DE SYMÉTRIE 2 (Miroir) --- \n")
print(f"L'atome de départ est le Gallium (Ga) aux coordonnées : {Ga_coord}\n")
print(f"Application de l'opération {S2.as_xyz_str()} :\n")
print(S2.rotation_matrix, " * ", Ga_coord, " + ", S2.translation_vector)
print(f"\nL'atome d'arrivée a pour coordonnées : {Ga_sym}\n")


# Visualisation avec JSmol
view2 = quick_view(conv_struc)
display(view2)

view2.script("reset")
view2.script("background white")
view2.script("unitcell on")
view2.script("axes on")
view2.script("spacefill 23%")
view2.script("wireframe 0.08")
view2.script("zoom 140")

view2.script("select aluminum; color lightgray")
view2.script("select gallium; color lightgreen")
view2.script("select nitrogen; color lightblue")

view2.script("draw delete")
view2.script("draw plane1 plane {0 1 0} {0 0 0}")
view2.script("color $plane1 translucent green")
view2.script("draw symop 9 {atomno=2}")
view2.script("select atomno=2; color red; spacefill 0.5")

--- OPÉRATION DE SYMÉTRIE 2 (Miroir) --- 

L'atome de départ est le Gallium (Ga) aux coordonnées : [0. 0. 0.]

Application de l'opération x, -y, z :

[[ 1.  0.  0.]
 [ 0. -1.  0.]
 [ 0.  0.  1.]]  *  [0. 0. 0.]  +  [0. 0. 0.]

L'atome d'arrivée a pour coordonnées : [0. 0. 0.]



JsmolView(layout=Layout(align_self='stretch', height='400px'))

#### Illustration du plan miroir

![Miroir](Images_symmetry/reflexion.png)

### 2.3 Roto-inversion sur l'Azote (N)

La troisième opération étudiée est une roto-inversion d'ordre 4 ($\bar{4}$), appliquée à l'atome d'Azote. Cette opération correspond à une rotation de $90^\circ$ autour de l'axe $z$ suivie d'une inversion par rapport au centre. Elle est décrite par l'équation matricielle suivante :

$$\begin{pmatrix} x' \\ y' \\ z' \end{pmatrix} = \begin{pmatrix} 0 & -1 & 0 \\ 1 & 0 & 0 \\ 0 & 0 & -1 \end{pmatrix} \begin{pmatrix} x \\ y \\ z \end{pmatrix} + \begin{pmatrix} 0 \\ 0 \\ 0 \end{pmatrix}$$

Comme au point 2.2, le déterminant de la matrice vaut $-1$, il s'agit donc donc d'une opération de deuxième espèce modifiant la chiralité du site.

L'application de cette roto-inversion à la position de l'atome d'Azote conduit à une nouvelle position distincte dans la maille. Cette position est néanmoins cristallographiquement équivalente, car elle peut être reliée à la position initiale par une translation de réseau. Cela confirme que cette opération appartient bien au groupe de symétrie du cristal.

In [4]:
print("--- OPÉRATION DE SYMÉTRIE 3 (Roto-inversion) --- \n")
print(f"L'atome de départ est l'Azote (N) aux coordonnées : {N_coord}\n")
print(f"Application de l'opération {S3.as_xyz_str()} :\n")
print(S3.rotation_matrix, " * ", N_coord, " + ", S3.translation_vector)
print(f"\nL'atome d'arrivée a pour coordonnées : {N_sym}\n")

# Visualisation avec JSmol
view3 = quick_view(conv_struc)
display(view3)

view3.script("reset")
view3.script("background white")
view3.script("unitcell on")
view3.script("axes on")
view3.script("spacefill 23%")
view3.script("wireframe 0.08")
view3.script("zoom 140")

view3.script("select aluminum; color lightgray")
view3.script("select gallium; color lightgreen")
view3.script("select nitrogen; color lightblue")
view3.script("select all")

view3.script("draw delete")
view3.script("draw symop 3 {atomno=5}")  
view3.script("select atomno=5; color red; spacefill 0.5")

--- OPÉRATION DE SYMÉTRIE 3 (Roto-inversion) --- 

L'atome de départ est l'Azote (N) aux coordonnées : [0.5        0.         0.73826481]

Application de l'opération -y, x, -z :

[[ 0. -1.  0.]
 [ 1.  0.  0.]
 [ 0.  0. -1.]]  *  [0.5        0.         0.73826481]  +  [0. 0. 0.]

L'atome d'arrivée a pour coordonnées : [ 0.          0.5        -0.73826481]



JsmolView(layout=Layout(align_self='stretch', height='400px'))

#### Illustration de la roto-inversion

![Roto-inversion](Images_symmetry/rotoinversion.png)